In [ ]:
Lab 4 Parte B

B) Filtros Passa-alta
Estude a teoria e implemente os códigos destes tutoriais para o seu entendimento de filtragem passa-alta de imagens. 
Atenção especial no entendimento das funções de filtragem: Laplacian(), Sobel(), e Canny().

Obs.: execute estes exercicios no computador local com Jupyter Notebook na linguagem Python e OpenCV.

(B.1) Image Gradients (OpenCV Tutorials)
https://docs.opencv.org/4.x/d5/d0f/tutorial_py_gradients.html

(B.2) Canny Edge Detection (OpenCV Tutorials)
https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html

Parte experimental:

1) Nesta etapa, utilize suas próprias imagens obtidas no Lab3 - imagem dos membros individuais e imagem dos avatar individual. 

    (a) Elabore um programa que realize as filtragens passa-alta (com os filtros Laplaciano, SobelX, SobelY, e Canny) nas suas imagens individuais e 
salve as imagens resultantes de cada filtragem, em formato .jpg.
    Objetivo é encontrar a melhor representação de bordas de cada pessoa e do avatar. 
    Na filtragem Sobel, teste pelo menos duas variações de kernel.
    Na filtragem Canny, teste pelo menos três variações de ajuste quanto ao limiar superior, limiar inferior, e grau de supressão.

In [ ]:
import cv2
import numpy as np
import os

# ============================================
# NOMES DAS IMAGENS DE ENTRADA
# ============================================

# caio_sem_avatar.jpg
# caio_com_avatar.jpg
# edson_sem_avatar.jpg
# edson_com_avatar.jpg
# nicolas_sem_avatar.jpg
# nicolas_com_avatar.jpg

# ============================================
# CONFIGURAÇÕES
# ============================================

# Lista de pessoas (nomes dos arquivos sem extensão)
pessoas = ['caio', 'edson', 'nicolas']

# Variações: 'sem_avatar' e 'com_avatar'
versoes = ['sem_avatar', 'com_avatar']

# Extensão das imagens (padrão .jpg)
extensao = '.jpg'

# --- Sobel: tamanhos de kernel a testar ---
# Valores possíveis: 1, 3, 5, 7 (ímpares)
sobel_kernels = [3, 5]  # Altere aqui para testar outros valores

# --- Canny: lista de dicionários com limiares ---
# Cada dicionário deve conter:
#   'threshold1': limiar inferior
#   'threshold2': limiar superior
#   'desc': descrição curta (usada no nome do arquivo)
# Altere os valores abaixo para testar diferentes combinações
canny_params = [
    {'threshold1': 30, 'threshold2': 90, 'desc': 'baixo'},
    {'threshold1': 100, 'threshold2': 200, 'desc': 'medio'},
    {'threshold1': 200, 'threshold2': 400, 'desc': 'alto'}
    # Adicione mais variações se desejar
]

# ============================================
# PROCESSAMENTO (arquivos na mesma pasta)
# ============================================

print("Iniciando processamento...")

for pessoa in pessoas:
    for versao in versoes:
        # Monta o nome do arquivo de entrada (ex: caio_sem_avatar.jpg)
        nome_entrada = f"{pessoa}_{versao}{extensao}"

        # Verifica se o arquivo existe no diretório atual
        if not os.path.exists(nome_entrada):
            print(f"Arquivo não encontrado: {nome_entrada}. Pulando...")
            continue

        # Carregar imagem
        img = cv2.imread(nome_entrada)
        if img is None:
            print(f"Erro ao ler: {nome_entrada}")
            continue

        # Converter para escala de cinza
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        print(f"Processando: {nome_entrada}")

        # ---------- Laplaciano ----------
        laplacian = cv2.Laplacian(gray, cv2.CV_64F)
        laplacian = np.uint8(np.abs(laplacian))
        nome_saida = f"laplacian_{pessoa}_{versao}.jpg"
        cv2.imwrite(nome_saida, laplacian)

        # ---------- Sobel (X e Y) para cada kernel ----------
        for k in sobel_kernels:
            # Sobel X
            sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=k)
            sobelx = np.uint8(np.abs(sobelx))
            nome_saida = f"sobelx_k{k}_{pessoa}_{versao}.jpg"
            cv2.imwrite(nome_saida, sobelx)

            # Sobel Y
            sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=k)
            sobely = np.uint8(np.abs(sobely))
            nome_saida = f"sobely_k{k}_{pessoa}_{versao}.jpg"
            cv2.imwrite(nome_saida, sobely)

        # ---------- Canny para cada conjunto de parâmetros ----------
        for params in canny_params:
            edges = cv2.Canny(gray, params['threshold1'], params['threshold2'])
            nome_saida = f"canny_{params['desc']}_{pessoa}_{versao}.jpg"
            cv2.imwrite(nome_saida, edges)

print("Processamento concluído!")

Todas as imagens encontram-se na pasta "lab_4_parte_B_1" (https://github.com/caiofalcheti/PDI-UFABC-1Q2026/tree/main/lab4/parte%20b/lab_4_parte_B_1)

2) Nesta segunda etapa, buscamos aumentar a Nitidez da imagem usando o Laplaciano, na imagem dos membros juntos do grupo e na imagem dos avatar juntos. 

    (a) Elabore um programa que realize a equação (3.54) do Gonzalez, g(x,y) = f(x,y) + c . { Laplac [ f(x,y) ] } com c = -1. Utilize tanto as imagens original,
quanto as imagens ruidosa obtida na seção (A).

In [ ]:
import cv2
import numpy as np
import os

# Lista de pessoas
pessoas = ['caio', 'edson', 'nicolas']

# Variações: sem_avatar e com_avatar
versoes = ['sem_avatar', 'com_avatar']

# Extensão das imagens
extensao = '.jpg'

# Tipos de ruído (para as imagens ruidosas)
# As imagens originais são apenas o nome base
# As ruidosas terão sufixos: _gauss e _sp
ruidos = ['gauss', 'sp']

# Parâmetro c da equação (Gonzalez recomenda c = -1 para a máscara padrão)
c = -1

# Tamanho do kernel para o Laplaciano (usar 3, pois é o padrão)
laplacian_ksize = 3

def add_gaussian_noise(image, mean=0, std=25):
    """Adiciona ruído Gaussiano à imagem."""
    noise = np.random.normal(mean, std, image.shape).astype(np.uint8)
    noisy_image = cv2.add(image, noise)
    return noisy_image

def add_salt_and_pepper_noise(image, noise_ratio=0.02):
    """Adiciona ruído Salt-and-Pepper à imagem."""
    noisy_image = image.copy()
    h, w, c = noisy_image.shape
    noisy_pixels = int(h * w * noise_ratio)

    for _ in range(noisy_pixels):
        row, col = np.random.randint(0, h), np.random.randint(0, w)
        if np.random.rand() < 0.5:
            noisy_image[row, col] = [0, 0, 0]       # pimenta (preto)
        else:
            noisy_image[row, col] = [255, 255, 255] # sal (branco)

    return noisy_image

# ============================================
# PROCESSAMENTO
# ============================================

print("Iniciando aumento de nitidez com Laplaciano...")

for pessoa in pessoas:
    for versao in versoes:
        # Nome base (original)
        nome_base = f"{pessoa}_{versao}{extensao}"
        
        # Verifica se a original existe
        if not os.path.exists(nome_base):
            print(f"Original não encontrada: {nome_base}. Pulando...")
            continue

        # Carrega imagem original
        img_original = cv2.imread(nome_base)
        if img_original is None:
            print(f"Erro ao ler {nome_base}")
            continue

        # Converte para escala de cinza (o Laplaciano é aplicado em cinza)
        gray_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2GRAY)

        # Calcula Laplaciano da original
        lap_original = cv2.Laplacian(gray_original, cv2.CV_64F, ksize=laplacian_ksize)

        # Aplica equação: g = f + c * Laplaciano
        # Como lap pode ter valores negativos, fazemos a conversão cuidadosamente
        sharp_original = gray_original + c * lap_original
        # Clipa para 0-255 e converte para uint8
        sharp_original = np.clip(sharp_original, 0, 255).astype(np.uint8)

        # Salva imagem nítida da original
        nome_saida = f"sharp_{pessoa}_{versao}.jpg"
        cv2.imwrite(nome_saida, sharp_original)
        print(f"Salvo: {nome_saida}")


        # Agora processa as imagens ruidosas (gauss e sp)
        for ruido in ruidos:
            nome_ruido = f"{pessoa}_{versao}_{ruido}{extensao}"
            if not os.path.exists(nome_ruido):
                print(f"Ruído não encontrado: {nome_ruido}. Pulando...")
                continue

            img_ruido = cv2.imread(nome_ruido)
            if img_ruido is None:
                print(f"Erro ao ler {nome_ruido}")
                continue

            gray_ruido = cv2.cvtColor(img_ruido, cv2.COLOR_BGR2GRAY)
            lap_ruido = cv2.Laplacian(gray_ruido, cv2.CV_64F, ksize=laplacian_ksize)
            sharp_ruido = gray_ruido + c * lap_ruido
            sharp_ruido = np.clip(sharp_ruido, 0, 255).astype(np.uint8)

            nome_saida = f"sharp_{pessoa}_{versao}_{ruido}.jpg"
            cv2.imwrite(nome_saida, sharp_ruido)
            print(f"Salvo: {nome_saida}")

print("Processamento concluído!")

Todas as imagens encontram-se na pasta "lab_4_parte_B_2" (https://github.com/caiofalcheti/PDI-UFABC-1Q2026/tree/main/lab4/parte%20b/lab_4_parte_B_2)

    (b) Elabore um novo programa que realize a operação Highboost, com pelo menos dois níveis de realce. Nesse caso utilize apenas a imagem original.

In [ ]:
import cv2
import numpy as np
import os

# Nomes das pessoas (conforme seus arquivos)
pessoas = ['caio', 'edson', 'nicolas']

# Variações: com e sem avatar
versoes = ['sem_avatar', 'com_avatar']

# Extensão das imagens
extensao = '.jpg'

# Parâmetros do Highboost
gaussian_kernel = 5          # Tamanho do kernel do filtro gaussiano (ímpar)
sigma = 0                     # Desvio padrão (0 = automático)
k_values = [1, 3]             # Fatores de realce (dois níveis)

# ============================================
# PROCESSAMENTO
# ============================================

print("Iniciando filtragem Highboost em imagens coloridas...")

for pessoa in pessoas:
    for versao in versoes:
        # Nome do arquivo de entrada (ex: caio_sem_avatar.jpg)
        nome_entrada = f"{pessoa}_{versao}{extensao}"

        if not os.path.exists(nome_entrada):
            print(f"Arquivo não encontrado: {nome_entrada}. Pulando...")
            continue

        # Carregar imagem colorida
        img_color = cv2.imread(nome_entrada)
        if img_color is None:
            print(f"Erro ao ler {nome_entrada}")
            continue

        # Converter para YUV (Y = luminância, U e V = crominância)
        img_yuv = cv2.cvtColor(img_color, cv2.COLOR_BGR2YUV)

        # Separar os canais
        y, u, v = cv2.split(img_yuv)

        # Converter Y para float32 para precisão
        y_float = np.float32(y)

        # Aplicar suavização gaussiana no canal Y
        blurred = cv2.GaussianBlur(y_float, (gaussian_kernel, gaussian_kernel), sigma)

        # Calcular a máscara (detalhes = Y original - borrado)
        mask = y_float - blurred

        # Aplicar para cada valor de k
        for k in k_values:
            # Highboost no canal Y: Y' = Y + k * mask
            y_sharp = y_float + k * mask

            # Recortar para o intervalo [0,255] e converter de volta para uint8
            y_sharp = np.clip(y_sharp, 0, 255).astype(np.uint8)

            # Recombinar os canais
            img_sharp_yuv = cv2.merge([y_sharp, u, v])

            # Converter de volta para BGR (colorido)
            img_sharp = cv2.cvtColor(img_sharp_yuv, cv2.COLOR_YUV2BGR)

            # Salvar a imagem resultante
            nome_saida = f"highboost_k{k}_{pessoa}_{versao}.jpg"
            cv2.imwrite(nome_saida, img_sharp)
            print(f"Salvo: {nome_saida}")

print("Processamento concluído!")

Todas as imagens encontram-se na pasta "lab_4_parte_B_3" (https://github.com/caiofalcheti/PDI-UFABC-1Q2026/tree/main/lab4/parte%20b/lab_4_parte_B_3)

3) Elabore um novo programa em que a imagem de entrada é da webcam, e que mostre o resultado da nitidez e do highboost em janelas opencv, de forma contínua na tela do computador.
Utilize a tecla [s] do teclado para permitir salvar a imagem sendo apresentada na tela, em formato .jpg. 

In [ ]:
import cv2
import numpy as np

# ============================================
# CONFIGURAÇÕES (altere aqui se desejar)
# ============================================
# Parâmetros para nitidez (Laplaciano)
c = -1                     # coeficiente da equação (Gonzalez)
laplacian_ksize = 3        # tamanho do kernel do Laplaciano

# Parâmetros para highboost
gaussian_kernel = 5        # tamanho do kernel do desfoque (ímpar)
sigma = 0                   # desvio padrão (0 = automático)
k_values = [1, 3]          # fatores de realce (dois níveis)

# ============================================
# FUNÇÕES DE PROCESSAMENTO (em espaço YUV)
# ============================================
def process_sharpness(frame, c, ksize):
    """Aplica nitidez via Laplaciano no canal Y."""
    # Converter para YUV
    yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    y, u, v = cv2.split(yuv)
    y_float = np.float32(y)

    # Laplaciano
    lap = cv2.Laplacian(y_float, cv2.CV_64F, ksize=ksize)

    # Equação: g = f + c * lap
    sharp_y = y_float + c * lap
    sharp_y = np.clip(sharp_y, 0, 255).astype(np.uint8)

    # Reconstituir imagem colorida
    sharp_yuv = cv2.merge([sharp_y, u, v])
    sharp_bgr = cv2.cvtColor(sharp_yuv, cv2.COLOR_YUV2BGR)
    return sharp_bgr

def process_highboost(frame, kernel_size, sigma, k):
    """Aplica highboost no canal Y com fator k."""
    yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    y, u, v = cv2.split(yuv)
    y_float = np.float32(y)

    # Borrar
    blurred = cv2.GaussianBlur(y_float, (kernel_size, kernel_size), sigma)

    # Máscara de detalhes
    mask = y_float - blurred

    # Highboost
    high_y = y_float + k * mask
    high_y = np.clip(high_y, 0, 255).astype(np.uint8)

    high_yuv = cv2.merge([high_y, u, v])
    high_bgr = cv2.cvtColor(high_yuv, cv2.COLOR_YUV2BGR)
    return high_bgr

# ============================================
# INICIALIZAÇÃO DA WEBCAM
# ============================================
cap = cv2.VideoCapture(0)  # 0 para webcam padrão
if not cap.isOpened():
    print("Erro ao abrir a webcam")
    exit()

print("Controles:")
print("  Pressione 's' para salvar as imagens atuais")
print("  Pressione 'q' para sair")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Falha na captura")
        break

    # Processar nitidez
    sharp = process_sharpness(frame, c, laplacian_ksize)

    # Processar highboost para k=1 e k=3
    high1 = process_highboost(frame, gaussian_kernel, sigma, k_values[0])
    high3 = process_highboost(frame, gaussian_kernel, sigma, k_values[1])

    # Exibir janelas
    cv2.imshow('Original', frame)
    cv2.imshow('Nitidez (Laplaciano)', sharp)
    cv2.imshow(f'Highboost (k={k_values[0]})', high1)
    cv2.imshow(f'Highboost (k={k_values[1]})', high3)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('s'):
        # Salvar as quatro imagens com timestamp
        timestamp = cv2.getTickCount()
        cv2.imwrite(f'original_{timestamp}.jpg', frame)
        cv2.imwrite(f'nitidez_{timestamp}.jpg', sharp)
        cv2.imwrite(f'highboost_k{k_values[0]}_{timestamp}.jpg', high1)
        cv2.imwrite(f'highboost_k{k_values[1]}_{timestamp}.jpg', high3)
        print("Imagens salvas!")

    elif key == ord('q'):
        break

# Liberar recursos
cap.release()
cv2.destroyAllWindows()